# Experiment

I had an idea of how to reduce the size of the output layer by rethinking the problem as predicting goals that the home team will score (between 0 and 10) and the goals that away team will score (between 0 and 10). I'm thinking that the output layer that will have 22 units. The idea is to 'combine' 2 softmax classifiers

In [2]:
import sys, os
sys.path.insert(0, "/home/roman/Code/AIEngineering/Projects/FIFACompetition/WorldCup2026/experiments")

In [21]:
# Imports
# Allowing notebook to import from components folder
import sys, os
root_path = os.path.abspath(os.path.join('..'))
if root_path not in sys.path:
    sys.path.insert(0, root_path)

import numpy as np
import numpy.typing as npt
import math
from components.data import load_original_data_with_trimmed_y, split_and_normalize_dataset, collect_player_data_into_sum_concat_teams_form_from_old
from components.player_modification import arrange_player_details_to_offense_defense

# Import data

I will change output labels to match to have 22 values then I will normalize targets

In [22]:
X, Y = load_original_data_with_trimmed_y()
print(Y.shape)

def convert_trimmed_y_to_reduced(
    Y: npt.NDArray
):
    """
    Convert a (m, 121) output labels to (m,22) reduced outputs

    Args:
        Y (ndarray): a (m, 121) array with 121 target labels

    Return:
        Y_reduced (ndarray): a (m, 22) array with 22 target labels
    """
    m = Y.shape[0]
    predictions = np.argmax(Y, axis=1)
    home_team_score_labels = []
    away_team_score_labels = []

    for i in range(m):
        home_team_score = np.zeros((11,))
        away_team_score = np.zeros((11,))
        home_score = math.floor(predictions[i] / 11)
        away_score = predictions[i] % 11
        home_team_score[home_score] = 1
        away_team_score[away_score] = 1

        home_team_score_labels.append(home_team_score)
        away_team_score_labels.append(away_team_score)

    home_team_score_labels = np.array(home_team_score_labels)
    away_team_score_labels = np.array(away_team_score_labels)
    reduced_labels = np.concatenate((home_team_score_labels, away_team_score_labels), axis=-1)
    return reduced_labels
    
X_offense_defense = collect_player_data_into_sum_concat_teams_form_from_old(Input=X, num_player_features=8, extract_player_vector=arrange_player_details_to_offense_defense)
Y_reduced = convert_trimmed_y_to_reduced(Y=Y)

(1400, 121)


In [24]:
# Normalize data and split data
X_train, X_cv, X_test, Y_train, Y_cv, Y_test = split_and_normalize_dataset(
    X=X_offense_defense, 
    Y=Y_reduced, 
    training_set_ratio=0.86, 
    cv_test_set_ratio=0.5, 
    random_state=121
)

print(X_train.shape, Y_train.shape)
print(X_cv.shape, Y_cv.shape)
print(X_test.shape, Y_test.shape)

(16, 1203) (22, 1203)
(16, 98) (22, 98)
(16, 99) (22, 99)
